## 08b — TrustyAI first capture (JSON only)

**Objective:** prove the KServe logger → TrustyAI path works. After this notebook, laptop `/info` must show `muffin-chihuahua` with `observations ≥ 1`.

### Why this notebook (not `08_trustyai_traffic.ipynb`)

TrustyAI rejects KServe inputs that include `"parameters": {"binary_data_size": …}`.
Binary `/infer` often returns **200** from OVMS, so a “fallback to JSON” never runs — and `/info` stays `{}`.

This notebook sends **JSON only**, no `parameters` field.

### Endpoint

| Port | Effect |
|------|--------|
| **`:9081`** | KServe **agent** / logger → TrustyAI ✅ |
| `:8888` | OVMS direct → **bypasses** TrustyAI ❌ |
| `:80` | Headless Service → Connection refused ❌ |

```text
http://muffin-chihuahua-predictor.chihuahua-vs-muffin-jan.svc.cluster.local:9081
```

### After running

On your laptop:

```bash
curl -sk -H "Authorization: Bearer $TOKEN" "$TRUSTY_ROUTE/info"
```


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import requests
from PIL import Image

MODEL = "muffin-chihuahua"
# Agent port only — logger copies /infer to TrustyAI
EP = "http://muffin-chihuahua-predictor.chihuahua-vs-muffin-jan.svc.cluster.local:9081"
SESSION = requests.Session()

# Keep small: each logged image stores a full 224×224×3 tensor (~2 MiB in TrustyAI)
N_PER_CLASS = 2
TEST_ROOT = Path("/opt/app-root/src/data/muffin-chihuahua/test")


def preprocess(path: Path) -> np.ndarray:
    img = Image.open(path).convert("RGB").resize((224, 224))
    arr = np.asarray(img).astype("float32") / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype="float32")
    std = np.array([0.229, 0.224, 0.225], dtype="float32")
    arr = (arr - mean) / std
    return np.transpose(arr, (2, 0, 1))[None, ...].astype(np.float32)


def infer_json(path: Path):
    """JSON /infer with no 'parameters' — required for TrustyAI logger parse."""
    chw = preprocess(path)
    payload = {
        "inputs": [
            {
                "name": "input",
                "shape": list(chw.shape),
                "datatype": "FP32",
                "data": chw.ravel().tolist(),
            }
        ]
    }
    r = SESSION.post(
        f"{EP}/v2/models/{MODEL}/infer",
        headers={"Content-Type": "application/json"},
        data=json.dumps(payload),
        timeout=120,
    )
    print(path.name, r.status_code, f"{r.elapsed.total_seconds():.2f}s")
    r.raise_for_status()
    return r


paths = []
paths += sorted((TEST_ROOT / "muffin").glob("*.jpg"))[:N_PER_CLASS]
paths += sorted((TEST_ROOT / "chihuahua").glob("*.jpg"))[:N_PER_CLASS]

if not paths:
    raise SystemExit(
        f"No images under {TEST_ROOT}. Run the dataset notebook first, or check the path."
    )

print("Sending", len(paths), "JSON inferences to", EP)
t0 = time.perf_counter()
for p in paths:
    infer_json(p)
print(f"Done in {time.perf_counter() - t0:.1f}s")
print("Next on laptop: curl -sk -H \"Authorization: Bearer $TOKEN\" \"$TRUSTY_ROUTE/info\"")